# Act IV — The Geographic Divide
### *The Diverging American Dream: Can America Afford a Family?*

**Story beat:** Acts I–III showed *what* diverged, *who* it hit, and *what it
costs* to start a family. Act IV asks the final question: *where* does
geography determine whether a family is economically possible?

| # | Visualization | Type | Requirement |
|---|---|---|---|
| 1 | County Childcare Burden Choropleth | **Map-based** | ✅ Map |
| 2 | State Childcare Burden Over Time | **Interactive temporal** | ✅ Interactive + Temporal |
| 3 | The American Dream Dashboard | **Dashboard** | ✅ Dashboard |
| 4 | Two Americas: 2008 vs. 2022 | **Static small multiples** | ✅ Static |

---


## Setup — Imports & Theme

In [9]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from theme import (
    THEME, C,
    apply_theme, apply_dark_theme,
    annotate_threshold, add_recession_bands, watermark,
    PLOTLY_LAYOUT
)

RAW = "/Users/ishika/Desktop/MS/Scholarship_Project/Data/"

print("✓ Imports complete")
print(f"  Red  (high burden) : {C.red}")
print(f"  Teal (affordable)  : {C.green}")
print(f"  Gold (reference)   : {C.gold}")


✓ Imports complete
  Red  (high burden) : #B84A2E
  Teal (affordable)  : #2A7C6F
  Gold (reference)   : #C8872A


## Load All Act IV Data

Same `load_fred()` pattern as Acts I–III. Childcare data from raw CSV.

In [10]:
def load_fred(filename, col_rename=None, start_year=1960, end_year=2025):
    """Load a FRED CSV — identical to Acts I, II, III."""
    df = pd.read_csv(RAW + filename, parse_dates=['DATE'])
    df['year'] = df['DATE'].dt.year
    val_col = [c for c in df.columns
               if c not in ['DATE', 'year']][0]
    df = df.rename(columns={val_col: col_rename or val_col})
    name = col_rename or val_col
    df = df.dropna(subset=[name])
    df = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    return df.groupby('year')[name].mean()


def parse_epi_year(d):
    """Handle 2-digit EPI dates — identical to Acts II, III."""
    yr = int(str(d).split('/')[-1])
    if yr < 100:
        return (2000 + yr) if yr <= 30 else (1900 + yr)
    return yr


# ── Childcare county data (DOL NDCP 2008-2022) ───────────────
print("Loading childcare county data...")
cc_raw = pd.read_csv(RAW + 'childcare_county.csv', low_memory=False)

print(f"  Shape  : {cc_raw.shape}")
print(f"  Years  : {sorted(cc_raw['STUDYYEAR'].unique())}")
print(f"  States : {cc_raw['STATE_ABBREVIATION'].nunique()}")

# Keep only columns we need
cc = cc_raw[[
    'STATE_NAME', 'STATE_ABBREVIATION', 'COUNTY_NAME',
    'COUNTY_FIPS_CODE', 'STUDYYEAR',
    'MCINFANT',    # median weekly cost, infant, center-based
    'MCTODDLER',   # median weekly cost, toddler
    'MCPRESCHOOL', # median weekly cost, preschool
    'MFI_2022',    # median family income, inflation-adjusted to 2022$
    'MHI_2022',    # median household income, 2022$
]].copy()

cc.columns = [
    'state_name', 'state', 'county', 'fips',
    'year', 'cost_infant_wk', 'cost_toddler_wk',
    'cost_preschool_wk', 'mfi_2022', 'mhi_2022'
]

# MCINFANT is WEEKLY — convert to annual
cc['cost_infant_annual']    = (cc['cost_infant_wk']    * 52).round(0)
cc['cost_toddler_annual']   = (cc['cost_toddler_wk']   * 52).round(0)
cc['cost_preschool_annual'] = (cc['cost_preschool_wk'] * 52).round(0)

# Childcare burden = annual infant cost / median family income
cc['infant_pct_income'] = (
    cc['cost_infant_annual'] / cc['mfi_2022'] * 100
).round(1)

# FIPS zero-padded to 5 digits
cc['fips'] = cc['fips'].apply(
    lambda x: str(int(x)).zfill(5) if pd.notna(x) else None
)

cc = cc.dropna(subset=['cost_infant_annual', 'mfi_2022', 'fips'])

latest_yr = int(cc['year'].max())
first_yr  = int(cc['year'].min())
cc_latest = cc[cc['year'] == latest_yr].copy().reset_index(drop=True)
cc_first  = cc[cc['year'] == first_yr ].copy().reset_index(drop=True)

print(f"\n  Clean rows       : {len(cc)}")
print(f"  Year range       : {first_yr}–{latest_yr}")
print(f"  Counties ({latest_yr}): {len(cc_latest)}")
print(f"  Sample annual infant cost: ${cc_latest['cost_infant_annual'].median():,.0f}/yr")
print(f"  Sample burden   : {cc_latest['infant_pct_income'].median():.1f}% of income")

# ── State aggregation ─────────────────────────────────────────
cc_state = (
    cc.groupby(['state', 'year'])
    .agg(
        median_cost=('cost_infant_annual', 'median'),
        median_pct =('infant_pct_income',  'median'),
        n_counties =('county', 'count'),
    )
    .round(1).reset_index()
)

# ── National aggregation ──────────────────────────────────────
cc_national = (
    cc.groupby('year')
    .agg(
        nat_median_pct =('infant_pct_income',  'median'),
        nat_median_cost=('cost_infant_annual', 'median'),
        pct_above_7    =('infant_pct_income',  lambda x: (x > 7).mean()  * 100),
        pct_above_15   =('infant_pct_income',  lambda x: (x > 15).mean() * 100),
    )
    .round(1).reset_index()
)

print("\n  National burden by year:")
print(cc_national[['year','nat_median_pct',
                   'pct_above_7','pct_above_15']].to_string(index=False))

# ── FRED series for dashboard ─────────────────────────────────
s_income = load_fred('MEFAINUSA672N.csv', 'real_median_fam_income')
s_cpi    = load_fred('CPIAUCSL.csv',      'cpi_all')
s_tfr    = load_fred('SPDYNTFRTINUSA.csv','tfr',         start_year=1960)
s_home   = load_fred('MSPUS.csv',         'median_home', start_year=1963)
s_gini   = load_fred('GINIALLRH.csv',     'gini',        start_year=1967)
s_wages  = load_fred('AHETPI.csv',        'wages',       start_year=1964)

aw = pd.read_csv(RAW + 'epi_annual_wages.csv')
aw['year'] = aw['date'].apply(parse_epi_year)
aw = aw.drop(columns='date').set_index('year').sort_index()
aw.columns = ['bottom_90','top_10','top_5','top_1','top_0_1',
              'p90_95','p95_99','p90_99','p99_999']

# Price-to-income ratio (nominal / nominal)
cpi_2023    = s_cpi.loc[2023]
mfi_nominal = s_income * (s_cpi / cpi_2023)
pti         = (s_home / mfi_nominal).round(2)

# Top 1% wage index (1979=100)
top1_idx = (aw['top_1'] / aw.loc[1979, 'top_1'] * 100).round(1) \
           if 1979 in aw.index else aw['top_1']

print("\n✓ All Act IV FRED data loaded")
for name, s in [
    ('Real median income', s_income),
    ('TFR',                s_tfr),
    ('Gini',               s_gini),
    ('Median home price',  s_home),
    ('Top 1% idx',         top1_idx),
]:
    print(f"  {name:25s}: {int(s.dropna().index.min())}–"
          f"{int(s.dropna().index.max())}  ({len(s.dropna())} obs)")


Loading childcare county data...
  Shape  : (48308, 370)
  Years  : [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
  States : 52

  Clean rows       : 35453
  Year range       : 2008–2022
  Counties (2022): 2661
  Sample annual infant cost: $9,025/yr
  Sample burden   : 11.6% of income

  National burden by year:
 year  nat_median_pct  pct_above_7  pct_above_15
 2008             9.3         87.4           4.1
 2009             9.1         84.3           5.3
 2010             9.2         83.3           4.4
 2011             9.6         86.0           4.9
 2012             9.9         89.4           5.5
 2013            10.3         92.9           7.1
 2014            10.6         94.5           8.8
 2015            10.6         94.3           7.9
 2016            10.5         94.7           6.6
 

---
## Visualization 1 — County Childcare Burden Choropleth
### Map-Based Visualization · Dropdown toggles % of income vs. annual cost

HHS affordability threshold = **7% of household income**.
The median U.S. county spends well above this on infant care alone.


In [16]:
# Shared color scale: teal (affordable) → rust red (crisis)
COLOR_SCALE = [
    [0.0,  '#e3f0ec'],
    [0.15, '#a8d4c4'],
    [0.30, '#fac775'],
    [0.50, '#e07b39'],
    [0.70, '#c87028'],
    [0.85, '#b84a2e'],
    [1.0,  '#8a2e18'],
]

GEOJSON = ('https://raw.githubusercontent.com/plotly/datasets'
           '/master/geojson-counties-fips.json')

# ── Pct-of-income choropleth ──────────────────────────────────
cc_state_latest = (
    cc[cc['year'] == latest_yr]
    .groupby(['state_name', 'state'])
    .agg(
        median_pct =('infant_pct_income',  'median'),
        median_cost=('cost_infant_annual', 'median'),
    )
    .round(1).reset_index()
)

cc_state_first = (
    cc[cc['year'] == first_yr]
    .groupby(['state_name', 'state'])
    .agg(
        median_pct =('infant_pct_income',  'median'),
        median_cost=('cost_infant_annual', 'median'),
    )
    .round(1).reset_index()
)

# ── Pct-of-income choropleth — STATE level ────────────────────
fig_pct = px.choropleth(
    cc_state_latest.dropna(subset=['median_pct']),
    locations='state',
    locationmode='USA-states',        # ← state abbreviations, not FIPS
    color='median_pct',
    color_continuous_scale=COLOR_SCALE,
    range_color=[5, 25],
    scope='usa',
    hover_name='state_name',
    hover_data={
        'state':       False,
        'median_pct':  ':.1f',
        'median_cost': ':,.0f',
    },
    labels={
        'median_pct':  'Infant care % of income',
        'median_cost': 'Median annual cost ($)',
    },
)

# ── Annual cost choropleth — STATE level ──────────────────────
fig_cost = px.choropleth(
    cc_state_latest.dropna(subset=['median_cost']),
    locations='state',
    locationmode='USA-states',
    color='median_cost',
    color_continuous_scale=COLOR_SCALE,
    range_color=[5000, 20000],
    scope='usa',
    hover_name='state_name',
    hover_data={
        'state':       False,
        'median_cost': ':,.0f',
        'median_pct':  ':.1f',
    },
    labels={
        'median_cost': 'Annual cost ($)',
        'median_pct':  '% of income',
    },
)


# ── Combine with dropdown ─────────────────────────────────────
fig_map = go.Figure()

n_pct  = len(fig_pct.data)
n_cost = len(fig_cost.data)

for t in fig_pct.data:
    t.visible = True
    fig_map.add_trace(t)
for t in fig_cost.data:
    t.visible = False
    fig_map.add_trace(t)

nat = cc_national[cc_national['year'] == latest_yr]
nat_pct  = nat['nat_median_pct'].values[0]  if len(nat) else 11.6
pct_7    = nat['pct_above_7'].values[0]     if len(nat) else 88
nat_cost = nat['nat_median_cost'].values[0] if len(nat) else 9100

dropdown_btns = [
    dict(
        label='Infant care as % of family income',
        method='update',
        args=[
            {'visible': [True]*n_pct + [False]*n_cost},
            {'title': {'text': (
                f'<b>Infant Childcare as % of Median Family Income · {latest_yr}</b><br>'
                f'<sup>HHS affordability threshold = 7% · '
                f'Median county: {nat_pct:.1f}% · '
                f'{pct_7:.0f}% of counties exceed the 7% threshold</sup>'
             ), 'font': dict(size=THEME["font_size"]["lg"],
                             family=THEME["font"])}},
        ],
    ),
    dict(
        label='Annual infant care cost ($)',
        method='update',
        args=[
            {'visible': [False]*n_pct + [True]*n_cost},
            {'title': {'text': (
                f'<b>Annual Infant Childcare Cost (Center-Based) · {latest_yr}</b><br>'
                f'<sup>National median: ${nat_cost:,.0f}/year</sup>'
             ), 'font': dict(size=THEME["font_size"]["lg"],
                             family=THEME["font"])}},
        ],
    ),
]

apply_theme(fig_map)
fig_map.update_layout(
    title=dict(
        text=(
            f'<b>Infant Childcare as % of Median Family Income · {latest_yr}</b><br>'
            f'<sup>HHS threshold = 7% · '
            f'Median county: {nat_pct:.1f}% · '
            f'{pct_7:.0f}% of counties exceed the 7% threshold</sup>'
        ),
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    geo=dict(
        scope='usa',
        showlakes=True,
        lakecolor='rgb(245,240,232)',
        bgcolor=THEME['background'],
        landcolor=THEME['background'],
    ),
    coloraxis=fig_pct.layout.coloraxis,
    coloraxis_colorbar=dict(
        title=dict(text='% of income',
                   font=dict(family=THEME['font_mono'], size=11)),
        tickfont=dict(family=THEME['font_mono'], size=10),
        ticksuffix='%',
    ),
    updatemenus=[dict(
        buttons=dropdown_btns,
        direction='down', showactive=True,
        x=0.01, xanchor='left',
        y=1.13, yanchor='top',
        bgcolor=THEME['background'], bordercolor=THEME['border'],
        font=dict(size=12, family=THEME['font_sans']),
    )],
    annotations=[dict(
        x=0.01, y=1.21, xref='paper', yref='paper',
        text='← Toggle view:',
        showarrow=False,
        font=dict(size=11, color=C.muted, family=THEME['font_mono']),
    )],
    height=560,
    margin=dict(t=140, b=20, l=20, r=20),
)

# Callout annotation
fig_map.add_annotation(
    x=0.98, y=0.06,
    xref='paper', yref='paper',
    text=(f'<b>{pct_7:.0f}%</b> of counties<br>'
          f'exceed the 7%<br>HHS threshold'),
    showarrow=False,
    font=dict(size=11, color=C.red, family=THEME['font_mono']),
    bgcolor=THEME['background'],
    bordercolor=C.red, borderwidth=1,
    align='center',
)

watermark(fig_map, (f'Source: U.S. Department of Labor · '
                    f'National Database of Childcare Prices (NDCP) · '
                    f'{latest_yr} · MFI inflation-adjusted to 2022$'))
fig_map.show()

# Print extremes
print(f"\n── County extremes ({latest_yr}) ───────────────────────────────────────")
print(f"  National median burden    : {nat_pct:.1f}% of income")
print(f"  National median annual $  : ${nat_cost:,.0f}/year")
print(f"  Counties above 7% (HHS)  : {pct_7:.0f}%")
print(f"  Counties above 15%        : {nat['pct_above_15'].values[0]:.0f}%")
print("\n  Most burdened counties:")
top5 = cc_latest.nlargest(5, 'infant_pct_income')[
    ['county','state','infant_pct_income','cost_infant_annual']]
for _, r in top5.iterrows():
    print(f"    {r['county']:25s}, {r['state']}: "
          f"{r['infant_pct_income']:.1f}%  (${r['cost_infant_annual']:,.0f}/yr)")



── County extremes (2022) ───────────────────────────────────────
  National median burden    : 11.6% of income
  National median annual $  : $9,025/year
  Counties above 7% (HHS)  : 96%
  Counties above 15%        : 19%

  Most burdened counties:
    Suffolk County           , MA: 29.9%  ($30,680/yr)
    Fresno County            , CA: 28.4%  ($21,611/yr)
    Bronx County             , NY: 27.6%  ($15,600/yr)
    Wahkiakum County         , WA: 27.2%  ($18,580/yr)
    Piute County             , UT: 27.0%  ($11,895/yr)


---
## Visualization 2 — State Childcare Burden Over Time
### Interactive Temporal Line Chart

State-level median infant childcare cost as % of family income, 2008–2022.
Gray lines = all states. Colored lines = selected highlights.
National median = thick dark reference line.
HHS 7% threshold = teal dashed line.


In [12]:
years_list  = sorted(cc_state['year'].unique())
states_list = sorted(cc_state['state'].unique())

fig_ts = go.Figure()

# HHS 7% threshold
fig_ts.add_hline(
    y=7, line_dash='dash', line_color=C.green, line_width=1.5,
    annotation_text='HHS affordability threshold (7%)',
    annotation_position='top right',
    annotation_font=dict(size=10, color=C.green, family=THEME['font_mono']),
)

# All state background lines
for state in states_list:
    s_data = cc_state[cc_state['state'] == state].sort_values('year')
    if len(s_data) < 3:
        continue
    fig_ts.add_trace(go.Scatter(
        x=s_data['year'].tolist(),
        y=s_data['median_pct'].tolist(),
        name=state,
        mode='lines',
        line=dict(color=C.red, width=1.0),
        opacity=0.25,
        hovertemplate=(f'<b>{state}</b> · %{{x}}<br>'
                       '%{y:.1f}% of income<extra></extra>'),
        showlegend=False,
    ))

# National median — thick reference
fig_ts.add_trace(go.Scatter(
    x=cc_national['year'].tolist(),
    y=cc_national['nat_median_pct'].tolist(),
    name='National median',
    mode='lines',
    line=dict(color='#0f0e0b', width=3.2, dash='dot'),
    hovertemplate='<b>National median</b> · %{x}<br>%{y:.1f}%<extra></extra>',
))

# Highlighted states
HIGHLIGHTS = [
    ('DC', 'D.C.',          C.red),
    ('MA', 'Massachusetts', C.orange),
    ('CA', 'California',    C.purple),
    ('MS', 'Mississippi',   C.green),
    ('TX', 'Texas',         C.blue),
]

for abbr, name, color in HIGHLIGHTS:
    s_data = cc_state[cc_state['state'] == abbr].sort_values('year')
    if len(s_data) == 0:
        continue
    fig_ts.add_trace(go.Scatter(
        x=s_data['year'].tolist(),
        y=s_data['median_pct'].tolist(),
        name=name,
        mode='lines+markers',
        line=dict(color=color, width=2.5),
        marker=dict(size=6, color=color),
        hovertemplate=(f'<b>{name}</b> · %{{x}}<br>'
                       '%{y:.1f}% of income<extra></extra>'),
    ))
    # End label
    last = s_data.iloc[-1]
    fig_ts.add_annotation(
        x=last['year'], y=last['median_pct'],
        text=f'<b>{abbr}</b> {last["median_pct"]:.1f}%',
        showarrow=False,
        xanchor='left', xshift=8,
        font=dict(size=10, color=color, family=THEME['font_mono']),
    )

add_recession_bands(fig_ts)
apply_theme(fig_ts)

fig_ts.update_layout(
    title=dict(
        text=('<b>State Childcare Burden Over Time, 2008–2022</b><br>'
              '<sup>Median infant center-based care as % of median family income · '
              'Gray = all states · Colored = selected highlights · '
              'Dark dashed = national median</sup>'),
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    xaxis=dict(
        title='Year',
        tickmode='linear', dtick=2,
        range=[years_list[0] - 0.3, years_list[-1] + 2.2],
    ),
    yaxis=dict(
        title='Infant childcare cost (% of family income)',
        ticksuffix='%',
        range=[0, cc_state['median_pct'].max() * 1.18],
    ),
    legend=dict(
        x=0.01, y=0.99,
        xanchor='left', yanchor='top',
        bgcolor='rgba(245,240,232,0.9)',
        font=dict(size=11),
    ),
    height=500,
    margin=dict(t=100, b=70, l=75, r=130),
)

watermark(fig_ts, ('Source: DOL National Database of Childcare Prices '
                   '(NDCP) · 2008–2022 · State medians from county data'))
fig_ts.show()

# Rankings
ls = cc_state[cc_state['year'] == latest_yr]
print(f"\n── State burden rankings ({latest_yr}) ──────────────────────────────────")
print("  Highest burden:")
for _, r in ls.nlargest(8,'median_pct').iterrows():
    print(f"    {r['state']}: {r['median_pct']:.1f}%  ${r['median_cost']:,.0f}/yr")
print("\n  Lowest burden:")
for _, r in ls.nsmallest(8,'median_pct').iterrows():
    print(f"    {r['state']}: {r['median_pct']:.1f}%  ${r['median_cost']:,.0f}/yr")



── State burden rankings (2022) ──────────────────────────────────
  Highest burden:
    CT: 19.4%  $16,640/yr
    CA: 18.2%  $18,374/yr
    DC: 17.9%  $25,480/yr
    MA: 17.8%  $18,304/yr
    WA: 17.3%  $17,034/yr
    AZ: 15.7%  $11,180/yr
    NY: 15.6%  $12,844/yr
    OK: 15.3%  $10,657/yr

  Lowest burden:
    KS: 6.7%  $4,971/yr
    SD: 8.0%  $6,240/yr
    WY: 9.1%  $8,242/yr
    GA: 9.3%  $5,642/yr
    MI: 9.6%  $7,052/yr
    IA: 9.8%  $8,302/yr
    ID: 9.8%  $6,602/yr
    TX: 10.0%  $7,488/yr


---
## Visualization 3 — The American Dream Dashboard
### Five KPI Sparkline Cards

A single-view summary of the five core metrics from all four acts.
Each card shows the current value, % change since base year, and a
full sparkline of the historical trajectory.


In [13]:
KPIS = [
    {
        'title':   'Real Median\nFamily Income',
        'series':  s_income,
        'base_yr': 1960,
        'fmt':     lambda v: f'${v:,.0f}',
        'color':   C.blue,
        'dir':     'up_good',
    },
    {
        'title':   'Home Price\n÷ Income',
        'series':  pti,
        'base_yr': 1963,
        'fmt':     lambda v: f'{v:.1f}×',
        'color':   C.red,
        'dir':     'down_good',  # higher = worse
    },
    {
        'title':   'Total\nFertility Rate',
        'series':  s_tfr,
        'base_yr': 1960,
        'fmt':     lambda v: f'{v:.2f}',
        'color':   C.gold,
        'dir':     'down_bad',   # lower = worse
    },
    {
        'title':   'Gini\nCoefficient',
        'series':  s_gini,
        'base_yr': 1967,
        'fmt':     lambda v: f'{v:.3f}',
        'color':   C.red,
        'dir':     'down_good',  # higher = worse
    },
    {
        'title':   'Top 1% Wages\n(1979 = 100)',
        'series':  top1_idx,
        'base_yr': 1979,
        'fmt':     lambda v: f'{v:.0f}',
        'color':   C.orange,
        'dir':     'context',
    },
]

fig_dash = make_subplots(
    rows=1, cols=5,
    subplot_titles=[k['title'].replace('\n',' ') for k in KPIS],
    horizontal_spacing=0.055,
)

for col_i, kpi in enumerate(KPIS, 1):
    s       = kpi['series'].dropna()
    curr    = float(s.iloc[-1])
    curr_yr = int(s.index.max())
    base_yr = kpi['base_yr'] if kpi['base_yr'] in s.index else int(s.index.min())
    base    = float(s.loc[base_yr])
    pct_chg = ((curr / base) - 1) * 100

    # Sparkline fill + line
    fig_dash.add_trace(go.Scatter(
        x=s.index.tolist(),
        y=s.values.tolist(),
        mode='lines',
        line=dict(color=kpi['color'], width=2),
        fill='tozeroy',
        fillcolor=kpi['color'] + '22',
        showlegend=False,
        hoverinfo='skip',
    ), row=1, col=col_i)

    # Current value — large number above the sparkline
    xref = 'x domain'     if col_i == 1 else f'x{col_i} domain'
    yref = 'y domain'     if col_i == 1 else f'y{col_i} domain'

    fig_dash.add_annotation(
        row=1, col=col_i,
        x=0.5, y=1.25,
        xref=xref, yref=yref,
        text=f'<b>{kpi["fmt"](curr)}</b>',
        showarrow=False,
        font=dict(size=17, color=kpi['color'], family=THEME['font']),
        bgcolor=THEME['background'],
    )

    # Delta label
    good = ((kpi['dir'] == 'up_good'  and pct_chg > 0) or
            (kpi['dir'] == 'down_bad' and pct_chg < 0))
    bad  = ((kpi['dir'] == 'up_good'   and pct_chg < 0) or
            (kpi['dir'] == 'down_good' and pct_chg > 0))
    dc   = C.teal if good else (C.red if bad else C.muted)
    sign = '+' if pct_chg > 0 else ''

    fig_dash.add_annotation(
        row=1, col=col_i,
        x=0.5, y=1.04,
        xref=xref, yref=yref,
        text=f'{sign}{pct_chg:.0f}% since {base_yr}',
        showarrow=False,
        font=dict(size=10, color=dc, family=THEME['font_mono']),
        bgcolor=THEME['background'],
    )

apply_theme(fig_dash)
fig_dash.update_layout(
    title=dict(
        text=('<b>The American Dream Dashboard</b><br>'
              '<sup>Five core metrics across six decades · '
              'Current value above each sparkline · '
              'Green delta = improved, red = worsened</sup>'),
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    height=340,
    showlegend=False,
    margin=dict(t=130, b=50, l=40, r=40),
)

# Clean up sparkline axes
for col_i in range(1, 6):
    fig_dash.update_yaxes(
        showticklabels=False, showgrid=False,
        zeroline=False, row=1, col=col_i,
    )
    fig_dash.update_xaxes(
        showticklabels=False, showgrid=False,
        zeroline=False, row=1, col=col_i,
    )

watermark(fig_dash, ('Sources: FRED (MEFAINUSA672N, MSPUS, SPDYNTFRTINUSA, '
                     'GINIALLRH) · EPI wage data · All values latest available'))
fig_dash.show()

print("\n── Dashboard numbers ─────────────────────────────────────────────────")
for kpi in KPIS:
    s       = kpi['series'].dropna()
    curr    = float(s.iloc[-1])
    base_yr = kpi['base_yr'] if kpi['base_yr'] in s.index else int(s.index.min())
    base    = float(s.loc[base_yr])
    pct_chg = ((curr / base) - 1) * 100
    sign    = '+' if pct_chg > 0 else ''
    title   = kpi['title'].replace('\n', ' ')
    print(f"  {title:32s}: {kpi['fmt'](curr):>10s}  "
          f"({sign}{pct_chg:.0f}% since {base_yr})")


ValueError: 
    Invalid value of type 'builtins.str' received for the 'fillcolor' property of scatter
        Received value: '#2A6FAF22'

    The 'fillcolor' property is a color and may be specified as:
      - A hex string (e.g. '#ff0000')
      - An rgb/rgba string (e.g. 'rgb(255,0,0)')
      - An hsl/hsla string (e.g. 'hsl(0,100%,50%)')
      - An hsv/hsva string (e.g. 'hsv(0,100%,100%)')
      - A named CSS color: see https://plotly.com/python/css-colors/ for a list

---
## Visualization 4 — Two Americas: 2008 vs. 2022
### Static Small Multiples

The same county map twice — earliest year and latest year in the NDCP.
**Identical color scale on both** so the darkening is directly visible.
The spread of red is the finding.


In [14]:
COLOR_SCALE_SM = [
    [0.0,  '#e3f0ec'], [0.15, '#a8d4c4'],
    [0.30, '#fac775'], [0.50, '#e07b39'],
    [0.70, '#c87028'], [0.85, '#b84a2e'],
    [1.0,  '#8a2e18'],
]
COLOR_RANGE_SM = [4, 22]

GEOJSON = ('https://raw.githubusercontent.com/plotly/datasets'
           '/master/geojson-counties-fips.json')

# Build both individual figs
fig_l = px.choropleth(
    cc_first.dropna(subset=['infant_pct_income']),
    geojson=GEOJSON, locations='fips',
    color='infant_pct_income',
    color_continuous_scale=COLOR_SCALE_SM,
    range_color=COLOR_RANGE_SM,
    scope='usa', hover_name='county',
    hover_data={'state': True, 'infant_pct_income': ':.1f', 'fips': False},
    labels={'infant_pct_income': '% of income', 'state': 'State'},
)
fig_r = px.choropleth(
    cc_latest.dropna(subset=['infant_pct_income']),
    geojson=GEOJSON, locations='fips',
    color='infant_pct_income',
    color_continuous_scale=COLOR_SCALE_SM,
    range_color=COLOR_RANGE_SM,
    scope='usa', hover_name='county',
    hover_data={'state': True, 'infant_pct_income': ':.1f', 'fips': False},
    labels={'infant_pct_income': '% of income', 'state': 'State'},
)

# Combine side by side
fig_both = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'{first_yr} — The baseline',
        f'{latest_yr} — The crisis deepened',
    ],
    specs=[[{'type': 'choropleth'}, {'type': 'choropleth'}]],
    horizontal_spacing=0.02,
)

for trace in fig_l.data:
    trace.showscale = False
    trace.coloraxis = 'coloraxis'
    fig_both.add_trace(trace, row=1, col=1)

for trace in fig_r.data:
    trace.showscale = False
    trace.coloraxis = 'coloraxis'
    fig_both.add_trace(trace, row=1, col=2)

# Compute stats for subtitle
med_f  = float(cc_first['infant_pct_income'].median())
med_l  = float(cc_latest['infant_pct_income'].median())
p7_f   = float((cc_first['infant_pct_income']  > 7).mean() * 100)
p7_l   = float((cc_latest['infant_pct_income'] > 7).mean() * 100)
p15_f  = float((cc_first['infant_pct_income']  > 15).mean() * 100)
p15_l  = float((cc_latest['infant_pct_income'] > 15).mean() * 100)

apply_theme(fig_both)
fig_both.update_layout(
    title=dict(
        text=(
            f'<b>Two Americas: Childcare Burden {first_yr} vs. {latest_yr}</b><br>'
            f'<sup>Infant center-based care as % of median family income · '
            f'Identical color scale on both maps · '
            f'Median: {med_f:.1f}% → {med_l:.1f}% · '
            f'Counties above 7%: {p7_f:.0f}% → {p7_l:.0f}%</sup>'
        ),
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    coloraxis=dict(
        colorscale=COLOR_SCALE_SM,
        cmin=COLOR_RANGE_SM[0],
        cmax=COLOR_RANGE_SM[1],
        colorbar=dict(
            title=dict(text='% of income',
                       font=dict(family=THEME['font_mono'], size=11)),
            tickfont=dict(family=THEME['font_mono'], size=10),
            ticksuffix='%',
            x=1.01,
        ),
    ),
    geo=dict(
        scope='usa', showlakes=True,
        lakecolor='rgb(245,240,232)',
        bgcolor=THEME['background'],
    ),
    geo2=dict(
        scope='usa', showlakes=True,
        lakecolor='rgb(245,240,232)',
        bgcolor=THEME['background'],
    ),
    height=420,
    margin=dict(t=120, b=20, l=10, r=80),
)

watermark(fig_both, (f'Source: DOL NDCP · {first_yr} and {latest_yr} · '
                     'Identical color scale enables direct comparison'))
fig_both.show()

print(f"\n── {first_yr} vs {latest_yr} comparison ──────────────────────────────────────")
print(f"  Median burden    : {med_f:.1f}% → {med_l:.1f}%  (+{med_l-med_f:.1f} pp)")
print(f"  Above 7% (HHS)  : {p7_f:.0f}% → {p7_l:.0f}% of counties")
print(f"  Above 15%        : {p15_f:.0f}% → {p15_l:.0f}% of counties")



── 2008 vs 2022 comparison ──────────────────────────────────────
  Median burden    : 9.3% → 11.6%  (+2.3 pp)
  Above 7% (HHS)  : 87% → 96% of counties
  Above 15%        : 4% → 19% of counties


---
## Act IV — Summary of Key Numbers

In [8]:
print("=" * 65)
print("ACT IV — KEY NUMBERS FOR WEBSITE CALLOUT CARDS")
print("=" * 65)

nat_l = cc_national[cc_national['year'] == latest_yr]
nat_f = cc_national[cc_national['year'] == first_yr]

print(f"\n── Childcare burden ({latest_yr}) ────────────────────────────────────")
if len(nat_l):
    print(f"  National median burden  : {nat_l['nat_median_pct'].values[0]:.1f}% of income")
    print(f"  National median annual  : ${nat_l['nat_median_cost'].values[0]:,.0f}/year")
    print(f"  Counties above 7% (HHS): {nat_l['pct_above_7'].values[0]:.0f}%")
    print(f"  Counties above 15%      : {nat_l['pct_above_15'].values[0]:.0f}%")
if len(nat_f) and len(nat_l):
    rise = (nat_l['nat_median_pct'].values[0]
            - nat_f['nat_median_pct'].values[0])
    print(f"  Rise {first_yr}→{latest_yr}          : +{rise:.1f} pp")

print(f"\n── Dashboard KPIs ────────────────────────────────────────────")
for kpi in KPIS:
    s       = kpi['series'].dropna()
    curr    = float(s.iloc[-1])
    base_yr = kpi['base_yr'] if kpi['base_yr'] in s.index else int(s.index.min())
    base    = float(s.loc[base_yr])
    pct_chg = ((curr / base) - 1) * 100
    sign    = '+' if pct_chg > 0 else ''
    title   = kpi['title'].replace('\n', ' ')
    print(f"  {title:32s}: {kpi['fmt'](curr):>10s}  "
          f"({sign}{pct_chg:.0f}% since {base_yr})")

print()
print("=" * 65)
print("All four Act IV visualizations complete.")
print("=" * 65)


ACT IV — KEY NUMBERS FOR WEBSITE CALLOUT CARDS

── Childcare burden (2022) ────────────────────────────────────
  National median burden  : 11.6% of income
  National median annual  : $9,025/year
  Counties above 7% (HHS): 96%
  Counties above 15%      : 19%
  Rise 2008→2022          : +2.3 pp

── Dashboard KPIs ────────────────────────────────────────────
  Real Median Family Income       :   $105,800  (+50% since 1976)
  Home Price ÷ Income             :       3.9×  (+14% since 1976)
  Total Fertility Rate            :       1.63  (-6% since 1976)
  Gini Coefficient                :      0.488  (+23% since 1976)
  Top 1% Wages (1979 = 100)       :        282  (+182% since 1979)

All four Act IV visualizations complete.


---
## Export Figures

In [ ]:
import os

OUT_DIR = '../outputs/act4/'
os.makedirs(OUT_DIR, exist_ok=True)

figures = {
    'act4_fig1_childcare_map':    fig_map,
    'act4_fig2_state_timeseries': fig_ts,
    'act4_fig3_dashboard':        fig_dash,
    'act4_fig4_two_americas':     fig_both,
}

for name, f in figures.items():
    html_path = OUT_DIR + name + '.html'
    f.write_html(
        html_path,
        include_plotlyjs='cdn',
        full_html=False,
        config={
            'displayModeBar': True,
            'displaylogo': False,
            'modeBarButtonsToRemove': ['lasso2d', 'select2d'],
            'toImageButtonOptions': {
                'format': 'png', 'filename': name, 'scale': 2,
            },
        },
    )
    print(f"  ✓ HTML: {html_path}")
    try:
        f.write_image(
            OUT_DIR + name + '.png',
            width=1200,
            height=f.layout.height or 520,
            scale=2,
        )
        print(f"  ✓ PNG : {OUT_DIR + name}.png")
    except Exception as e:
        print(f"  ⚠ PNG skipped — {e}")

print(f"\n✅  Act IV export complete → {OUT_DIR}")
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(OUT_DIR + fn)
    print(f"   {fn:50s}  {sz/1024:6.1f} KB")
